# Episode 8 — One Function Is a Synth: Parameters and Presets

⏱ About 25 minutes · Difficulty ★★☆

**Musical element this episode controls: all of them** — pitch, timbre, brightness, articulation and movement, together for the first time.

**The Max / RNBO equivalent:** a complete RNBO patch with a row of `param` objects on top and osc → filter → VCA inside, plus two envelopes.

**What you will hear:** three presets —
1. `pluck`: short, bright at the start
2. `pad`: a long note that fades in and slowly brightens
3. `wow`: a bass whose filter and wavetable open and close together, "wowww"

Then a melody played with this synth.

**By the end you can:**
- store a set of parameters in a `dict`
- read every line of `synth()` and say which episode it comes from
- change parameters and build your own presets
- play a series of notes with one preset

There is no new DSP in this episode, only "patching" and one new Python tool: the `dict`.

## Toolbox

Everything built in earlier episodes, copied as-is. Just run it; no need to re-read (the tricky lines carry `#` comments).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sr = 44100

def play(x, gain=0.3):
    # np.clip holds anything beyond ±1 so nothing blows up
    return Audio(np.clip(gain * np.asarray(x), -1, 1), rate=sr, normalize=False)

def scope(x, title="", seconds=0.03):
    n = int(sr * seconds)                      # only plot this many samples from the start
    plt.figure(figsize=(9, 2.5))
    plt.plot(np.arange(n) / sr, np.asarray(x)[:n])
    plt.xlabel("time (s)")
    plt.title(title)
    plt.show()

def make_table(harmonic_amps, N=2048):
    p = np.arange(N) / N                       # phase of every table position, 0 to 1
    table = np.zeros(N)                        # start from an empty table of zeros
    for k, amp in enumerate(harmonic_amps, start=1):
        # harmonic k: k cycles across the table, at level amp
        table += amp * np.sin(2 * np.pi * k * p)
    return table / np.max(np.abs(table))       # normalize: peak becomes 1

def make_frames(F=16, max_harmonics=32, N=2048):
    frames = []
    for f in range(F):
        # how many harmonics frame f gets: rising evenly from 1 to max_harmonics
        n_h = 1 + round(f * (max_harmonics - 1) / (F - 1))
        frames.append(make_table([1 / k for k in range(1, n_h + 1)], N))
    return np.array(frames)                    # stack into a 2-D array: F rows × N columns

def osc_wavetable(frames, phase, pos):
    F, N = frames.shape

    # --- which two frames are we between ---
    fpos = np.clip(pos, 0, 1) * (F - 1)        # turn 0..1 into a frame number 0..F-1, decimals allowed
    f0 = np.floor(fpos).astype(int)            # the frame below
    f1 = np.minimum(f0 + 1, F - 1)             # the frame above (never past the last one)
    ffrac = fpos - f0                          # how far above the lower frame, 0..1

    # --- which two positions inside a table (same as Episode 3) ---
    p = phase * N
    i0 = np.floor(p).astype(int) % N           # left position
    i1 = (i0 + 1) % N                          # right position, wrapping to 0 at the end
    frac = p - np.floor(p)                     # the fractional part

    # interpolate inside each frame first, then between the two frames
    a = (1 - frac) * frames[f0, i0] + frac * frames[f0, i1]
    b = (1 - frac) * frames[f1, i0] + frac * frames[f1, i1]
    return (1 - ffrac) * a + ffrac * b

def adsr(t, a, d, s, r, gate):
    a, d, r = max(a, 1e-4), max(d, 1e-4), max(r, 1e-4)    # times must not be 0, or we would divide by zero below
    # while held: (0,0) → (a,1) → (a+d,s), then stays at s
    held = np.interp(t, [0, a, a + d], [0, 1, s])
    # how high the envelope is at the moment of release (attack may not be finished)
    level_at_release = np.interp(gate, [0, a, a + d], [0, 1, s])
    # fall from that level to 0 over r seconds; np.clip stops it going negative
    released = level_at_release * np.clip(1 - (t - gate) / r, 0, 1)
    return np.where(t < gate, held, released)  # held before release, released after

def lowpass(x, cutoff, res=0.0):
    x = np.asarray(x, dtype=float)
    n_samples = len(x)
    # cutoff may be one number or an array: make it an array as long as the sound
    cutoff = np.clip(np.broadcast_to(cutoff, n_samples), 20, sr * 0.45)

    # --- turn cutoff / res into the filter's internal coefficients (one set per sample) ---
    g = np.tan(np.pi * cutoff / sr)
    k = 1.414 * (1 - res) + 0.05 * res         # damping: higher res → smaller k → stronger resonance
    a1 = 1 / (1 + g * (g + k))
    a2 = g * a1
    a3 = g * a2

    # .tolist() turns arrays into plain lists: much faster to read one by one in a loop
    xs, A1, A2, A3 = x.tolist(), a1.tolist(), a2.tolist(), a3.tolist()
    y = [0.0] * n_samples
    ic1 = ic2 = 0.0                            # two state variables = two history objects in gen~

    for n in range(n_samples):                 # one sample at a time, in order
        v3 = xs[n] - ic2
        v1 = A1[n] * ic1 + A2[n] * v3
        v2 = ic2 + A2[n] * ic1 + A3[n] * v3    # v2 is the lowpass output
        ic1 = 2 * v1 - ic1                     # update the state for the next sample
        ic2 = 2 * v2 - ic2
        y[n] = v2
    return np.array(y)

def phasor(freq, n_samples):
    # freq / sr = how far phase advances during one sample
    # cumsum adds the steps up; % 1 keeps the fractional part
    return np.cumsum(np.broadcast_to(freq, n_samples) / sr) % 1

frames = make_frames()

## 1. Signal flow

```
note ──► mtof ──► phasor ──► wavetable osc ──► lowpass ──► × ──► out
                                 ▲                ▲         ▲
                              wt_pos           cutoff    amp env
                                 ▲                ▲
                                 └──── mod env ───┘
```

Two envelopes:

- **amp env**: controls loudness (Episode 5)
- **mod env**: pushes both the wavetable position and the cutoff (Episode 7), each with its own amount

## 2. `dict`: parameters with names

Until now every parameter was its own variable (`cutoff = 300`, `res = 0.5` ...). With many parameters that gets messy.

In RNBO each `param` has a **name** and a **value**. Python's equivalent is the `dict`: a table of **name → value**. Max has a `dict` object of the same name, and it is the same thing.

- create: `{"name": value, "name": value}`
- read: `p["name"]`
- change: `p["name"] = new_value`

In [ ]:
p = {"note": 45, "cutoff": 300, "res": 0.2}

print("the whole dict :", p)
print("p['cutoff']    :", p["cutoff"])

p["cutoff"] = 1200                             # turning the cutoff knob
print("after change   :", p)

for name in p:                                 # take each name in turn
    print(name, "=", p[name])

There is another spelling, `dict(note=45, cutoff=300)`, with exactly the same result; the names need no quotes, so it is quicker to type. The presets below use it.

## 3. `mtof`

Max's `mtof`: MIDI note number → Hz.

- MIDI 69 = A4 = 440 Hz
- every 12 semitones up, the frequency doubles (Episode 7's `2 ** octaves`)

In [ ]:
def mtof(m):
    return 440 * 2 ** ((m - 69) / 12)          # (m - 69) / 12 = octaves away from A4

for m in [69, 57, 45, 33, 60]:
    print("MIDI", m, "->", round(mtof(m), 2), "Hz")

## 4. Building it step by step

Rather than presenting the final version outright, we build it in three steps and listen after each.

### Step 1: oscillator + amp envelope

Only 6 parameters are used: `note`, `wt_pos`, `amp_a`, `amp_d`, `amp_s`, `amp_r`.

The function's second and third inlets are `dur` (total length) and `gate` (time held), both with defaults.

**You should hear:** an A2 note with no filter; the timbre never changes, only the loudness moves.

In [ ]:
def synth_v1(p, dur=2.0, gate=1.2):
    n = int(sr * dur)
    t = np.arange(n) / sr
    amp_env = adsr(t, p["amp_a"], p["amp_d"], p["amp_s"], p["amp_r"], gate)
    phase = phasor(mtof(p["note"]), n)                     # note → Hz → phase
    x = osc_wavetable(frames, phase, p["wt_pos"])
    return x * amp_env

test = dict(note=45, wt_pos=0.6,
            amp_a=0.01, amp_d=0.3, amp_s=0.5, amp_r=0.3)

play(synth_v1(test))

### Step 2: add the filter

Two more parameters: `cutoff` and `res`. Just one extra line.

**You should hear:** the same note, duller (cutoff 600 Hz) with a little resonance.

In [ ]:
def synth_v2(p, dur=2.0, gate=1.2):
    n = int(sr * dur)
    t = np.arange(n) / sr
    amp_env = adsr(t, p["amp_a"], p["amp_d"], p["amp_s"], p["amp_r"], gate)
    phase = phasor(mtof(p["note"]), n)
    x = osc_wavetable(frames, phase, p["wt_pos"])
    x = lowpass(x, p["cutoff"], p["res"])                  # the new line
    return x * amp_env

test = dict(test, cutoff=600, res=0.4)                     # copy test and add two new parameters
play(synth_v2(test))

`dict(test, cutoff=600, res=0.4)`: copy `test` and add (or override) the items listed after it. The original `test` is untouched — although here we store the result back under the name `test`.

### Step 3: add the mod envelope

Six more parameters:

- `mod_a`, `mod_d`, `mod_s`, `mod_r`: the second envelope
- `wt_env`: how much it pushes the wavetable position (0–1)
- `cut_env`: how much it pushes the cutoff (octaves)

`wt_pos` and `cutoff` are no longer fixed numbers but **the knob value plus what the envelope adds**. That is exactly what Episode 7 did.

This is the final version, with 14 parameters in total.

In [ ]:
def synth(p, dur=2.0, gate=1.2):
    n = int(sr * dur)
    t = np.arange(n) / sr

    # two envelopes
    amp_env = adsr(t, p["amp_a"], p["amp_d"], p["amp_s"], p["amp_r"], gate)
    mod_env = adsr(t, p["mod_a"], p["mod_d"], p["mod_s"], p["mod_r"], gate)

    # oscillator: the wt_pos knob plus what the mod env adds
    phase = phasor(mtof(p["note"]), n)
    x = osc_wavetable(frames, phase, p["wt_pos"] + p["wt_env"] * mod_env)

    # filter: the cutoff knob, pushed up by cut_env octaves by the mod env
    x = lowpass(x, p["cutoff"] * 2 ** (p["cut_env"] * mod_env), p["res"])

    return x * amp_env                         # VCA

Line by line:

| line | what it does | episode |
|---|---|---|
| `t = ...` | the time table | 1 |
| `amp_env`, `mod_env` | two `adsr~` | 5 |
| `phase = phasor(mtof(...))` | `mtof` → `phasor~` | 7 |
| `osc_wavetable(...)` | position = knob + amount × mod env | 4, 7 |
| `lowpass(...)` | cutoff = knob, pushed up `cut_env` octaves by the mod env | 6, 7 |
| `x * amp_env` | the VCA (`*~`) | 5 |

### The 14 parameters

| name | meaning | suggested range |
|---|---|---|
| `note` | MIDI pitch | 28–60 |
| `wt_pos` | wavetable position | 0–1 |
| `wt_env` | amount of mod env → wt_pos | 0–1 |
| `cutoff` | filter cutoff (Hz) | 80–8000 |
| `res` | resonance | 0–0.9 |
| `cut_env` | amount of mod env → cutoff (octaves) | 0–5 |
| `amp_a` `amp_d` `amp_s` `amp_r` | amp envelope | times in seconds; `amp_s` 0–1 |
| `mod_a` `mod_d` `mod_s` `mod_r` | mod envelope | same |

## 5. Presets

A preset is simply a filled-in `dict`. `presets` is a "dict of dicts": name → preset.

In [ ]:
presets = {
    "pluck": dict(note=45, wt_pos=0.6, wt_env=0.0, cutoff=300, res=0.2, cut_env=4.0,
                  amp_a=0.005, amp_d=0.30, amp_s=0.0, amp_r=0.10,
                  mod_a=0.001, mod_d=0.15, mod_s=0.0, mod_r=0.10),
    "pad":   dict(note=57, wt_pos=0.2, wt_env=0.5, cutoff=800, res=0.1, cut_env=2.0,
                  amp_a=0.60, amp_d=0.30, amp_s=0.8, amp_r=0.70,
                  mod_a=1.00, mod_d=0.50, mod_s=0.6, mod_r=0.50),
    "wow":   dict(note=33, wt_pos=0.3, wt_env=0.7, cutoff=150, res=0.6, cut_env=4.5,
                  amp_a=0.01, amp_d=0.10, amp_s=0.9, amp_r=0.10,
                  mod_a=0.35, mod_d=0.40, mod_s=0.2, mod_r=0.20),
}

### A small "look at the preset" tool

Look before listening: `show(p)` draws the preset's two envelopes and the actual cutoff curve over time — like the envelope panels of Serum's interface.

In [ ]:
def show(p, dur=2.0, gate=1.2):
    t = np.arange(int(sr * dur)) / sr
    amp_env = adsr(t, p["amp_a"], p["amp_d"], p["amp_s"], p["amp_r"], gate)
    mod_env = adsr(t, p["mod_a"], p["mod_d"], p["mod_s"], p["mod_r"], gate)
    fig, axes = plt.subplots(1, 2, figsize=(11, 2.5))
    axes[0].plot(t, amp_env, label="amp env")
    axes[0].plot(t, mod_env, label="mod env")
    axes[0].axvline(gate, linestyle="--", color="gray")
    axes[0].legend()
    axes[0].set_xlabel("time (s)")
    axes[1].plot(t, p["cutoff"] * 2 ** (p["cut_env"] * mod_env))      # the same cutoff formula as inside synth()
    axes[1].set_ylabel("cutoff (Hz)")
    axes[1].set_xlabel("time (s)")
    plt.show()

### pluck

**You should hear:** a short plucked note with a bright "tick" at the start that quickly dulls and dies away.

**In the plot:** the mod env is shorter than the amp env, so the note goes dull first and quiet afterwards. The cutoff falls from 4800 Hz to 300 Hz within 0.15 seconds.

In [ ]:
show(presets["pluck"])
play(synth(presets["pluck"]))

### pad

**You should hear:** a long mid-range note that fades in while slowly brightening, then fades away gently after release.

**In the plot:** both envelopes have long attacks; the mod env takes 1 second to reach the top, so brightness changes more slowly than loudness.

In [ ]:
show(presets["pad"])
play(synth(presets["pad"]))

### wow

**You should hear:** a very low bass whose tone opens and closes like a mouth: "wowww".

**In the plot:** the amp env reaches the top almost instantly and stays; the mod env rises over 0.35 seconds and falls again — the shape of the "wow" is the shape of the mod env.

In [ ]:
show(presets["wow"])
play(synth(presets["wow"]))

## 6. Turning knobs

"Turning a knob" in Python is not live: **change a number → recompute → listen again**.

The upside: a loop can turn it for you, so you hear several positions of one knob in a row.

### `cutoff`

**You should hear:** the same `pluck` four times, brighter each time.

In [ ]:
for value in [100, 300, 900, 2700]:
    print("cutoff =", value)
    display(play(synth(dict(presets["pluck"], cutoff=value))))      # copy pluck, replacing only cutoff

### `cut_env`

**You should hear:** `wow` four times. With amount 0 there is no "wow" at all (the filter stays put); the larger the amount, the wider it opens.

In [ ]:
for value in [0, 1.5, 3, 4.5]:
    print("cut_env =", value)
    display(play(synth(dict(presets["wow"], cut_env=value))))

### `mod_a`

**You should hear:** `wow` four times. The opening speeds range from very fast ("wa") to very slow ("wooooow").

In [ ]:
for value in [0.02, 0.15, 0.35, 0.8]:
    print("mod_a =", value)
    display(play(synth(dict(presets["wow"], mod_a=value))))

## 7. Playing notes with a preset

Like the bassline in Episode 5: the loop swaps in a different `note` each time, renders a short sound, and everything is joined at the end.

**You should hear:** an 8-note arpeggio in A minor, played with the `pluck` sound.

In [ ]:
def play_notes(p, notes, dur=0.3):
    out = []
    for m in notes:
        one = synth(dict(p, note=m), dur=dur, gate=dur * 0.6)      # new pitch, everything else unchanged
        out.append(one)
    return np.concatenate(out)

arp = [45, 52, 57, 60, 64, 60, 57, 52]         # A2 E3 A3 C4 E4 C4 A3 E3
play(play_notes(presets["pluck"], arp))

Another preset, the same notes:

**You should hear:** the same notes, each with the "wow" tone (`mod_a` is shortened here; otherwise it would not open within 0.3 seconds).

In [ ]:
fast_wow = dict(presets["wow"], mod_a=0.08, mod_d=0.12)
play(play_notes(fast_wow, [33, 33, 40, 33, 36, 33, 43, 40]), gain=0.25)

## 8. Making your own preset

Start from an existing preset and change a few values. Below is a starting point: edit numbers, re-run, look at the plot, listen.

If you are short of ideas, aim for one of these:

- **more flute-like:** very low `wt_pos`, `cut_env` 0, `amp_a` around 0.08
- **more "laser":** `cut_env` 5, `mod_d` 0.08, `res` 0.8
- **more string-like:** `wt_pos` 0.8, `amp_a` 0.3, `cutoff` 2000

In [ ]:
my_patch = dict(presets["wow"],
                note=21,
                res=0.85,
                cut_env=5.0)

show(my_patch)
play(synth(my_patch), gain=0.2)

**You should hear (before editing):** the same `wow`, one octave lower, with stronger resonance and a more exaggerated "wow".

## Common mistakes

| Symptom | Cause |
|---|---|
| `KeyError: 'cutof'` | A misspelt name. The `dict` has no such entry |
| `KeyError: 'mod_a'` | The 6-parameter `test` was passed to `synth()`, which needs all 14 |
| A preset was changed but the sound was not | The cell calling `synth()` was not re-run |
| Very quiet or no sound | A very low `cutoff` with `cut_env` 0; or `amp_s` 0 with a very short decay |
| Junk tones at high pitches | `note` is too high; the 32 harmonics in the wavetable exceed sr / 2 (Episode 3) |

## Exercises

**1.** Make an `organ` preset: instant on, instant off, unchanging timbre. Hint: set both `wt_env` and `cut_env` to 0.

<details><summary>Answer</summary>

```python
organ = dict(note=57, wt_pos=0.4, wt_env=0.0, cutoff=3000, res=0.0, cut_env=0.0,
             amp_a=0.005, amp_d=0.01, amp_s=1.0, amp_r=0.01,
             mod_a=0.01, mod_d=0.01, mod_s=0.0, mod_r=0.01)
play(synth(organ))
```
</details>

**2.** Use a loop to hear `pad` with `res` at 0, 0.4 and 0.8.

<details><summary>Answer</summary>

```python
for value in [0, 0.4, 0.8]:
    display(play(synth(dict(presets["pad"], res=value)), gain=0.2))
```
</details>

**3.** Transpose the arpeggio `arp` up 5 semitones and play it. Hint: add 5 to every note inside a loop.

<details><summary>Answer</summary>

```python
higher = []
for m in arp:
    higher.append(m + 5)
play(play_notes(presets["pluck"], higher))
```
</details>

## Recap

| Max / RNBO | Python |
|---|---|
| an RNBO patch | the `synth(p)` function |
| a `param` object | one entry in a `dict` |
| the `dict` object | `dict` |
| a preset | a `dict` |
| `mtof` | `mtof(m)` |
| turning a knob | `dict(old, name=new_value)`, then call `synth()` again |

Having to "recompute" for every knob turn sounds clumsier than Max. But it buys an ability that is hard to get in Max.

**Next episode:** letting the computer turn the knobs and generate a pile of sounds in one go.